# 03. 쿠폰 A/B 테스트

- **분석 배경**
  - 장바구니에 상품을 담고 이탈한 고객에게 할인 쿠폰을 보내면 구매로 이어질지, 쿠폰 할인 비용을 감안해도 이익이 되는지 알고자 합니다.
  - 모든 고객에게 쿠폰을 보내기 전에 고객을 무작위로 두 집단으로 나눠 한 집단에만 쿠폰을 보내는 A/B 테스트로 효과를 확인합니다.
- **분석 대상**
  - 온라인몰에서 장바구니에 상품을 담고 이탈한 고객입니다.
  - 소규모 파일럿(2025-08-25 ~ 08-28 배정)과 본 테스트(2025-09-15 ~ 09-30 배정)는 서로 다른 고객입니다.
- **분석 질문**
  - 장바구니 고객에게 3만 원 이상 결제 시 5천 원 할인 쿠폰을 보내면 구매가 늘어나는가?
  - 구매가 늘어난다면 쿠폰을 발송하는 것은 회사에 이익이 되는가?
- **사용 데이터**
  - `ab_coupon.csv`(고객 10,000명)
- **분석 단위**: 고객

**학습 목표**

1. 무작위 배정을 확인하고, 모비율 차이 검정과 독립표본 t-검정으로 두 집단을 비교할 수 있다.
2. 표본 크기·검정력·제1종/제2종 오류를 파일럿과 본 테스트 결과와 연결해 설명할 수 있다.
3. '통계적으로 유의하다'와 '사업적으로 이익이다'를 구분해 판단할 수 있다.

## 0. 준비

### 라이브러리 불러오기

In [ ]:
# 필요한 모듈을 임포트합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
import hds
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

### 그래프 한글 폰트 설정

구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.<br>
음수 부호(−)가 네모로 깨지지 않도록 하는 설정도 함께 처리합니다.

In [ ]:
# 구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.
# 처음 실행할 때만 인터넷 연결이 필요하고, 이후에는 내려받은 파일을 다시 사용합니다.
hds.plot.set_font(font_name='Gowun Batang')

# 그래프 제목을 굵게 표시합니다.
plt.rc(group='axes', titleweight='bold')

### 데이터 불러오기

In [ ]:
# 쿠폰 A/B 테스트 csv 파일을 읽고 ab_test에 할당합니다.
ab_test = pd.read_csv(filepath_or_buffer='../data/ab_coupon.csv', encoding='utf-8-sig')

# ab_test의 처음 5행과 마지막 5행을 확인합니다.
# [참고] 데이터프레임 이름만 실행하면 처음 5행과 마지막 5행, 행 개수와 열 개수를 함께 보여줍니다.
ab_test

In [ ]:
# ab_test의 정보를 확인합니다.
ab_test.info()

## 1. 실험 설계 확인

**실험 설계**

| 항목 | 내용 |
|---|---|
| 대상 | 온라인몰에서 장바구니에 상품을 담고 장바구니에서 이탈한 고객 |
| 처리 | B집단에만 '3만 원 이상 구매 시 5천 원 할인' 쿠폰 발송 (A집단은 미발송) |
| 배정 | 하루 500명을 A/B에 **무작위**로 250명씩 배정 |
| 배정 기간 | 파일럿 4일(2025-08-25 ~ 08-28), 본 테스트 16일(2025-09-15 ~ 09-30) |
| 관찰 기간 | 배정일(쿠폰 발송일)로부터 14일 |
| 평가지표 1 | `purchased_14d`(구매전환율): 14일 안에 구매한 고객의 비율 |
| 평가지표 2 | `revenue_14d`(고객당 매출): 미구매 고객은 0원으로 포함 |

파일럿(소규모 사전 테스트)과 본 테스트는 **서로 다른 고객**입니다.

**질문**: 단계(phase)와 집단별로 몇 명이 배정되었고, 배정 기간은 언제인가?

In [ ]:
# 단계와 집단별 고객 수, 배정 시작일, 배정 종료일을 확인합니다.
ab_test.groupby(by=['phase', 'group']) \
    .agg(고객수=('customer_id', 'count'), 배정시작=('assign_date', 'min'), 배정종료=('assign_date', 'max'))

**해석**
- 파일럿과 본 테스트의 두 집단 인원이 모두 같습니다.
- 본 테스트는 파일럿보다 인원이 훨씬 많은데, 그 이유는 3절 표본 크기 산정에서 확인합니다.

## 2. 파일럿 분석

### 무작위 배정 확인

**질문**: 파일럿의 A와 B 집단은 고객 구성(신규/기존, 앱/웹)이 비슷한가?

In [ ]:
# ab_test에서 파일럿 행만 남기고 ab_pilot에 할당합니다.
ab_pilot = ab_test.loc[ab_test['phase'].eq(other='파일럿'), :]

# 집단별 신규/기존 비율을 교차표로 생성합니다.
pd.crosstab(index=ab_pilot['group'], columns=ab_pilot['tenure'], normalize='index', margins=True)

In [ ]:
# 집단별 앱/웹 비율을 교차표로 생성합니다.
pd.crosstab(index=ab_pilot['group'], columns=ab_pilot['device'], normalize='index', margins=True)

**해석**
- 두 집단의 신규/기존 비율과 앱/웹 비율이 거의 같습니다.
  - 무작위 배정이 잘 되었다면 쿠폰 말고는 두 집단이 다를 이유가 없습니다.
  - 고객을 두 집단으로 무작위 배정해야 결과 차이를 쿠폰 효과로 볼 수 있습니다.

**질문**: 1절 실험 설계처럼 고객을 무작위로 배정했습니다.<br>
배정이 잘 되었는지 두 집단의 고객 구성을 카이제곱 검정으로 확인합니다.<br>
가설검정 기초 실습에서 배운 교차분석입니다.

In [ ]:
# 집단과 tenure(신규/기존 고객 구분)의 카이제곱 검정을 실행합니다.
expected, observed, result = pg.chi2_independence(data=ab_pilot, x='group', y='tenure')

# 카이제곱 검정 결과를 확인합니다.
result.loc[[0], :]

In [ ]:
# 집단과 device(접속 기기)의 카이제곱 검정을 실행합니다.
expected, observed, result = pg.chi2_independence(data=ab_pilot, x='group', y='device')

# 카이제곱 검정 결과를 확인합니다.
result.loc[[0], :]

**해석**
- 두 검정 모두 유의확률이 0.05보다 크므로 집단 간 고객 구성이 다르다고 볼 근거가 부족합니다.
  - 즉 무작위 배정을 점검한 결과 **문제가 있다는 신호는 없습니다.**
  - [참고] 이 점검은 배정이 잘못된 경우를 걸러 내는 것이지 무작위 배정을 증명하지는 않으므로, 실무에서는 배정 기록(로그)과 배정 수도 함께 확인합니다.
- 만약 여기서 차이가 나왔다면, 결과를 비교하기 전에 배정 과정부터 점검해야 합니다.

### 구매전환율 비교

**질문**: 파일럿에서 집단별 구매전환율과 95% 신뢰구간은 얼마인가?

In [ ]:
# 집단별 고객 수, 구매자 수를 계산하고 ab_pilot_rate에 할당합니다.
ab_pilot_rate = ab_pilot.groupby(by='group')['purchased_14d'].agg(func=['count', 'sum'])

# ab_pilot_rate의 컬럼명을 '고객수'와 '구매자수'로 변경합니다.
ab_pilot_rate.columns = ['고객수', '구매자수']

# ab_pilot_rate에 집단별 전환율을 추가합니다.
ab_pilot_rate['전환율'] = ab_pilot_rate['구매자수'] / ab_pilot_rate['고객수']

# ab_pilot_rate에 집단별 전환율의 95% 신뢰구간을 추가합니다.
# [참고] proportion_confint()는 statsmodels.stats.proportion 모듈에 포함되어 있습니다.
# [참고] method='wilson'은 Wilson 구간으로, 기본값인 정규근사(Wald) 구간보다 포함 확률이 정확합니다.
# [참고] alpha를 생략하면 기본값 0.05가 적용되어 95% 신뢰구간을 계산합니다.
ci_lower, ci_upper = proportion_confint(count=ab_pilot_rate['구매자수'], nobs=ab_pilot_rate['고객수'], method='wilson')
ab_pilot_rate['95% CI 하한'], ab_pilot_rate['95% CI 상한'] = ci_lower, ci_upper

# ab_pilot_rate를 소수점 넷째 자리까지 확인합니다.
ab_pilot_rate.round(decimals=4)

**해석**
- B집단의 전환율이 A집단보다 높게 나왔지만, 두 집단의 신뢰구간이 많이 겹칩니다.
  - 집단당 인원이 적어서 구간이 넓기 때문입니다.

**질문**: 두 집단의 전환율 차이는 통계적으로 유의한가? (모비율 차이 검정)

In [ ]:
# B집단(처리군), A집단(대조군) 순서로 구매자 수를 리스트로 생성합니다.
count = ab_pilot_rate.loc[['B', 'A'], '구매자수'].tolist()

# B집단(처리군), A집단(대조군) 순서로 고객 수를 리스트로 생성합니다.
nobs = ab_pilot_rate.loc[['B', 'A'], '고객수'].tolist()

# 모비율 차이 검정을 실행합니다.
# [참고] proportions_ztest()는 statsmodels.stats.proportion 모듈에 포함되어 있습니다.
z_stat, p_value = proportions_ztest(count=count, nobs=nobs)

# 전환율 차이(B - A)를 계산합니다.
diff = count[0] / nobs[0] - count[1] / nobs[1]

# 전환율 차이의 95% 신뢰구간을 계산합니다.
# [참고] confint_proportions_2indep()는 statsmodels.stats.proportion 모듈에 포함되어 있습니다.
# [참고] 기본값 method='newcomb'는 각 집단의 Wilson 구간을 결합한 방법으로, Wald 방법보다 포함 확률이 정확합니다.
ci_lower, ci_upper = confint_proportions_2indep(count1=count[0], nobs1=nobs[0], count2=count[1], nobs2=nobs[1])

# 검정 결과를 출력합니다.
print(f'z = {z_stat:.3f}, 유의확률 = {p_value:.4f}')
print(f'전환율 차이(B − A) = {diff*100:.2f}%p')
print(f'95% 신뢰구간: {ci_lower*100:.2f} ~ {ci_upper*100:.2f}%p')

**해석**
- 유의확률이 0.05보다 크고, 차이의 신뢰구간에 0이 들어 있으므로 '쿠폰이 전환율을 높인다'고 **결론 내릴 수 없습니다.**
  - 이것은 '효과가 없다'는 뜻이 아닙니다.
- 실제로 차이가 있더라도 이 정도 인원으로는 우연과 구분할 수 없다는 뜻입니다.

**질문**: 파일럿 인원으로는 전환율이 최소 몇 %p 올라야 잡아낼 수 있었는가? (`hds.stat.ab_mde()`)

In [ ]:
# 파일럿의 집단별 인원을 계산합니다.
ab_pilot_n = ab_pilot.shape[0] // 2

# ab_pilot_n을 출력합니다.
print(f'파일럿의 집단별 인원: {ab_pilot_n}명')

In [ ]:
# 파일럿 인원으로 잡아낼 수 있는 최소 검출 효과(MDE)를 계산합니다.
# [참고] metric='proportion'은 비율을 기준으로 MDE를 계산함을 의미합니다.(기본값: 'proportion')
# baseline=0.10은 기준 전환율 10%, n_control=ab_pilot_n은 집단별 인원을 파일럿 인원으로 지정함을 의미합니다.
hds.stat.ab_mde(metric='proportion', baseline=0.10, n_control=ab_pilot_n)

**해석**
- `mde`는 검정력 0.8 기준에서 실제로 효과가 있을 때 80%의 확률로 잡아낼 수 있는 **최소 개선폭**입니다.
  - 위 표에서 `mde`가 0.0406이며, `baseline`에 `mde`를 더한 `treatment`가 중요합니다.
  - `treatment`는 처리군의 **실제 구매전환율**이 최소한 이 수준은 되어야 한다는 것을 의미합니다.
  - 처리군의 실제 구매전환율이 0.1406 이상일 때 같은 실험을 여러 번 반복하면 그중 80% 이상에서 귀무가설을 기각하는 결과가 나옵니다.
- 파일럿에서 기대했던 2%p는 `mde`보다 훨씬 작습니다.
  - 파일럿은 집단별 인원이 적어 효과를 확인하기 어려운 설계였다는 것을 뜻합니다.
  - 실험 전에 '이 인원으로 얼마나 작은 효과까지 볼 수 있는가'를 먼저 확인하면 이런 실수를 피할 수 있습니다.
- 현업에서는 표본 크기를 줄이려고 기대 효과를 근거 없이 크게 잡는 경우가 많습니다.
  - 만약 기준 전환율이 10%이고 목표 전환율을 14%로 설정했다면, 각 집단의 표본 크기는 약 1,000명으로 충분합니다. [파일럿 결과 참고]
- 그런데 목표 전환율 수준을 설정할 때 근거가 필요합니다.
  - 만약 실제 효과(구매전환율 차이)가 실험 전 기대보다 작다면 검정력이 낮아지게 됩니다.
  - 검정력이 낮으면(예를 들어 0.1), 제2종 오류($\beta=0.9$)가 커지므로 실제 효과가 있더라도 귀무가설을 기각하기 어렵게 됩니다.
- 다음 절에서는 기대 효과가 2%p일 때 쿠폰 발송 효과를 확인하려면 집단별 실험 인원으로 몇 명이 필요한지 계산합니다.

## 3. 표본 크기 산정

**질문**: 전환율 10%가 12%로 2%p 오르는 효과를 유의수준 0.05, 검정력 0.8로 잡아내려면 집단별로 몇 명이 필요한가?

In [ ]:
# 두 비율의 차이를 표준화한 효과 크기(Cohen's h)를 계산합니다.
# [참고] proportion_effectsize()는 statsmodels.stats.proportion 모듈에 포함되어 있습니다.
effect_size = proportion_effectsize(prop1=0.12, prop2=0.10)

# 효과 크기를 확인합니다.
print(f'효과 크기(h) = {effect_size:.4f}')

In [ ]:
# 표본 크기와 검정력을 계산할 객체를 생성합니다.
# [참고] NormalIndPower 클래스는 statsmodels.stats.power 모듈에 포함되어 있습니다.
power_analysis = NormalIndPower()

# 집단별 필요 인원을 계산합니다.
n_per_group = power_analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.8, ratio=1, alternative='two-sided')

# 집단별 필요 인원을 출력합니다.
print(f'집단별 필요 인원 = {n_per_group:,.0f}명')

**해석**
- 본 테스트는 집단별 인원을 이 값보다 조금 더 많은 4천 명으로 잡았습니다.
  - 전환율 2%p처럼 **작은 차이를 확인하려면 파일럿보다 많은 인원이 필요**합니다.

**질문**: 실제 효과가 10% → 12%라면, 파일럿 인원과 본 테스트 인원에서 그 효과를 잡아낼 확률(검정력)은 각각 얼마인가?

In [ ]:
# 본 테스트의 집단별 인원을 계산합니다.
ab_main_n = ab_test['phase'].eq(other='본테스트').sum() // 2

# 파일럿 테스트의 집단별 인원에 따른 검정력을 계산합니다.
power_pilot = power_analysis.power(effect_size=effect_size, nobs1=ab_pilot_n, alpha=0.05)

# 본 테스트의 집단별 인원에 따른 검정력을 계산합니다.
power_main = power_analysis.power(effect_size=effect_size, nobs1=ab_main_n, alpha=0.05)

# 검정력 결과를 출력합니다.
print(f'파일럿(집단별 {ab_pilot_n:,}명) 검정력: {power_pilot:.3f}')
print(f'본 테스트(집단별 {ab_main_n:,}명) 검정력: {power_main:.3f}')

**해석**
- 파일럿의 검정력이 0.299로 낮으므로 실제로 쿠폰 발송 효과가 있어도 '유의하지 않음'이 나올 가능성이 더 큰 설계였습니다.
- 본 테스트의 검정력은 0.8을 넘기 때문에 실제로 효과가 있다면 10번 중 8번은 차이를 잡아낼 수 있습니다.
- 여기서 계산한 검정력은 목표 효과(10% → 12%)로 실험 전에 계산하는 **사전 검정력**입니다.
  - t-검정 결과 표의 `power`처럼 관측한 효과 크기로 계산하는 사후 검정력과는 다릅니다.

**[참고] 진짜 전환율에 따라 달라지는 검정력**

- 검정력($1-\beta$)은 **진짜 효과가 얼마인지 정해야** 계산할 수 있습니다.<br>
$\beta$는 진짜 효과가 있는데도 관측된 차이가 우연히 작게 나와 귀무가설을 기각하지 못할 확률입니다. [제2종 오류]
- 파일럿 인원(집단별 1,000명)에서 쿠폰 집단의 진짜 전환율에 따라 $\beta$와 검정력($1-\beta$)은 다음과 같이 달라집니다.

  | 쿠폰 집단의 진짜 전환율 | $\beta$ (효과를 놓칠 확률) | 검정력 |
  |:-:|:-:|:-:|
  | 12% (기대 효과) | 0.70 | 0.30 |
  | 14.06% (파일럿의 MDE 수준) | 0.20 | 0.80 |

  [참고] 진짜 전환율이 14.06%처럼 충분히 높아야, 같은 실험을 여러 번 반복했을 때 80%에서 유의한 결과가 나옵니다.<br>
기대한 12%에서는 효과가 있어도 열 번 중 일곱 번은 놓칩니다.

### hds 라이브러리 함수로 상황별 표본 크기 계산하기

위 계산은 효과 크기를 구하고, 검정력 계산 객체를 만들고, 필요 인원을 푸는 세 단계를 거칩니다.<br>
`hds.stat.ab_sample_size()`는 `baseline`(**현재 수준**)과 `lift`(**기대 개선폭**)을 입력하면 이 과정을 한 번에 계산합니다.<br>
결과는 `item`(항목), `value`(값), `description`(설명) 열의 표로 반환합니다.

**질문**: 위와 같은 조건(전환율 10% → 12%)을 `hds.stat.ab_sample_size()`로 계산하면 결과가 같은가?

In [ ]:
# 전환율 10%가 12%로 오르는 효과를 잡아내는 데 필요한 인원을 계산합니다.
# [참고] metric='proportion'은 모비율 차이 검정을 의미합니다.(기본값: 'proportion')
# baseline=0.10은 기준 전환율 10%, lift=0.02는 기대 전환율 상승분 2%p를 의미합니다.
# [참고] lift=0.2, relative=True로 지정하면 '현재 전환율의 20% 개선'처럼 상대 비율로 입력할 수 있습니다.
hds.stat.ab_sample_size(metric='proportion', baseline=0.10, lift=0.02)

**해석**
- `effect_size`(효과 크기)가 위에서 계산한 Cohen's h와 같고, 집단별 인원은 위 결과를 올림한 값입니다.
  - 사람 수는 소수점이 있을 수 없으므로 항상 올림합니다.

**질문**: 쿠폰 금액을 3천 원과 5천 원 두 종류로 나누어 대조군과 함께 세 집단을 비교하면?

In [ ]:
# 대조군을 포함해 세 집단을 비교할 때 필요한 인원을 계산합니다.
# [참고] n_groups=3은 대조군을 포함해 세 집단을 비교함을 의미합니다.(기본값: 2)
hds.stat.ab_sample_size(metric='proportion', baseline=0.10, lift=0.02, n_groups=3)

**해석**
- 처리군마다 대조군과 한 번씩, 모두 두 번 비교하므로 유의수준 0.05를 2로 나눈 0.025로 낮춥니다. [Bonferroni 보정]
  - 비교 횟수가 늘면 우연히 유의하게 나올 가능성도 커지기 때문입니다.
- `treatment`는 두 처리군이 각각 대조군보다 최소한 이 수준은 되어야 한다는 것을 의미합니다.
  - 두 처리군이 같은 전환율을 낼 것이라는 가정은 아닙니다.
- 집단 수가 늘어나는 것에 더해 각 집단에 속한 인원도 늘어나므로 처리군은 꼭 필요한 만큼만 만듭니다.

## 4. 본 테스트 분석

### 무작위 배정 확인

**질문**: 본 테스트의 A와 B 집단은 고객 구성(신규/기존, 앱/웹)이 비슷한가?

In [ ]:
# ab_test에서 본 테스트 행만 남기고 ab_main에 할당합니다.
ab_main = ab_test.loc[ab_test['phase'].eq(other='본테스트'), :].copy()

# 집단별 신규/기존 비율을 교차표로 생성합니다.
pd.crosstab(index=ab_main['group'], columns=ab_main['tenure'], normalize='index', margins=True)

In [ ]:
# 집단별 앱/웹 비율을 교차표로 생성합니다.
pd.crosstab(index=ab_main['group'], columns=ab_main['device'], normalize='index', margins=True)

**해석**
- 본 테스트도 두 집단의 신규/기존 비율과 앱/웹 비율이 거의 같습니다.

**질문**: 파일럿과 마찬가지로 배정이 잘 되었는지 고객 구성을 카이제곱 검정으로 확인합니다.

In [ ]:
# 집단과 tenure(신규/기존 고객 구분)의 카이제곱 검정을 실행합니다.
expected, observed, result = pg.chi2_independence(data=ab_main, x='group', y='tenure')

# 카이제곱 검정 결과를 확인합니다.
result.loc[[0], :]

In [ ]:
# 집단과 device(접속 기기)의 카이제곱 검정을 실행합니다.
expected, observed, result = pg.chi2_independence(data=ab_main, x='group', y='device')

# 카이제곱 검정 결과를 확인합니다.
result.loc[[0], :]

**해석**
- 두 검정 모두 유의확률이 0.05보다 크므로 집단 간 고객 구성이 다르다고 볼 근거가 부족합니다.
  - 파일럿과 마찬가지로 배정에 문제가 있다는 신호는 없지만, 이것으로 무작위 배정을 증명하지는 않습니다.

### 구매전환율 비교

**질문**: 본 테스트에서 집단별 구매전환율과 차이 검정 결과는 어떤가?

In [ ]:
# 집단별 고객 수, 구매자 수를 계산하고 ab_main_rate에 할당합니다.
ab_main_rate = ab_main.groupby(by='group')['purchased_14d'].agg(func=['count', 'sum'])

# ab_main_rate의 컬럼명을 '고객수'와 '구매자수'로 변경합니다.
ab_main_rate.columns = ['고객수', '구매자수']

# ab_main_rate에 집단별 전환율을 추가합니다.
ab_main_rate['전환율'] = ab_main_rate['구매자수'] / ab_main_rate['고객수']

# ab_main_rate에 집단별 전환율의 95% 신뢰구간을 추가합니다.
ci_lower, ci_upper = proportion_confint(count=ab_main_rate['구매자수'], nobs=ab_main_rate['고객수'], method='wilson')
ab_main_rate['95% CI 하한'], ab_main_rate['95% CI 상한'] = ci_lower, ci_upper

# ab_main_rate를 소수점 넷째 자리까지 확인합니다.
ab_main_rate.round(decimals=4)

In [ ]:
# B집단(처리군), A집단(대조군) 순서로 구매자 수를 리스트로 생성합니다.
count = ab_main_rate.loc[['B', 'A'], '구매자수'].tolist()

# B집단(처리군), A집단(대조군) 순서로 고객 수를 리스트로 생성합니다.
nobs = ab_main_rate.loc[['B', 'A'], '고객수'].tolist()

# 모비율 차이 검정을 실행합니다.
z_stat, p_value = proportions_ztest(count=count, nobs=nobs)

# 전환율 차이(B - A)를 계산합니다.
diff = count[0] / nobs[0] - count[1] / nobs[1]

# 전환율 차이의 95% 신뢰구간을 계산합니다.
# [참고] 기본값 method='newcomb'는 각 집단의 Wilson 구간을 결합한 방법으로, Wald 방법보다 포함 확률이 정확합니다.
ci_lower, ci_upper = confint_proportions_2indep(count1=count[0], nobs1=nobs[0], count2=count[1], nobs2=nobs[1])

# 검정 결과를 출력합니다.
print(f'z = {z_stat:.3f}, 유의확률 = {p_value:.4f}')
print(f'전환율 차이(B − A) = {diff*100:.2f}%p')
print(f'95% 신뢰구간: {ci_lower*100:.2f} ~ {ci_upper*100:.2f}%p')

**해석**
- 유의확률이 0.05보다 훨씬 작고, 전환율 차이의 신뢰구간에 0이 들어 있지 않습니다.
  - **쿠폰을 받은 고객의 구매전환율이 더 높다**고 판단합니다.
- 전환율 차이 자체는 파일럿과 비슷한 크기인데, 인원이 늘어 신뢰구간이 좁아지면서 유의해졌습니다.

**질문**: 두 번째 지표인 고객당 매출(미구매 고객 0원 포함)은 어떤 분포인가?

In [ ]:
# 매출이 0원인 고객의 비율을 확인합니다.
print(f'매출 0원 고객 비율: {ab_main['revenue_14d'].eq(other=0).mean():.1%}')

In [ ]:
# 집단별 고객당 매출의 기술통계량을 확인합니다.
ab_main.groupby(by='group')['revenue_14d'].describe().round(decimals=0)

**해석**
- 대부분의 고객이 구매하지 않아 매출이 0원입니다.
  - 75% 분위수까지 0원이고, 평균은 구매한 소수의 고객이 만듭니다.
- 평균과 표준편차를 비교하면 표준편차가 평균보다 훨씬 큽니다.
  - 크게 치우친 분포라는 뜻입니다.

**질문**: 구매한 고객만 보면 매출 분포는 어떤 모양인가?

In [ ]:
# 그래프 눈금을 읽기 쉽도록 매출을 천 원 단위로 바꿉니다.
ab_main['revenue_k'] = ab_main['revenue_14d'] / 1000

# 천 원 단위 매출의 최솟값과 최댓값을 확인합니다.
ab_main['revenue_k'].agg(func=['min', 'max'])

In [ ]:
# 구매한 고객만 필터링하여 ab_main_buyers에 할당합니다.
ab_main_buyers = ab_main.loc[ab_main['revenue_k'].gt(other=0), :]

# 구매한 고객의 기술통계량을 확인합니다.
ab_main_buyers['revenue_k'].describe()

In [ ]:
# 1행 2열의 그래프 영역을 생성합니다.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4), dpi=100)

# 전체 고객의 매출 히스토그램을 10(천 원) 간격으로 그립니다.
sns.histplot(data=ab_main, x='revenue_k', binrange=(0, 400), bins=40, facecolor='silver', ax=axes[0])
axes[0].set(title='고객당 매출 (미구매 0원 포함)', xlabel='14일 매출 (천 원)', ylabel='고객 수')

# 구매 고객의 매출 분포를 집단별 커널 밀도 추정 곡선으로 그립니다.
# [참고] common_norm=False는 집단마다 곡선 아래 면적이 1이 되도록 하여 고객 수가 달라도 모양을 비교할 수 있게 합니다.
sns.kdeplot(data=ab_main_buyers, x='revenue_k', hue='group', fill=True, common_norm=False, ax=axes[1])
axes[1].set(title='구매 고객의 매출', xlabel='14일 매출 (천 원)', ylabel='밀도')
plt.show()

**해석**
- 왼쪽 그림은 0원에 거대한 막대가 있고 나머지는 오른쪽으로 길게 퍼져 있습니다.
  - 미구매 고객을 0원으로 포함한 매출은 흔히 이런 'ㄴ'자 형태를 보입니다.
- 오른쪽 그림은 구매 고객만 필터링하여 그린 커널 밀도 추정 곡선입니다.
  - 두 집단의 매출 분포가 비슷한 모양이며, 오른쪽 꼬리가 깁니다.

**질문**: 독립표본 t-검정 전에 집단별 고객당 매출의 정규성과 등분산성을 확인합니다.

In [ ]:
# 집단별 고객당 매출의 정규성 검정을 실행합니다.
pg.normality(data=ab_main, dv='revenue_14d', group='group')

In [ ]:
# 집단별 고객당 매출의 등분산성 검정을 실행합니다.
# [참고] center='mean'은 각 값과 집단 평균의 거리로 분산을 비교함을 의미합니다.(기본값: 'median')
pg.homoscedasticity(data=ab_main, dv='revenue_14d', group='group', method='levene', center='mean')

**해석**
- 정규성 검정의 `normal`이 False이므로 정규분포를 따른다고 보기 어렵습니다.
  - 0원이 대부분인 분포이니 당연한 결과입니다.
  - 표본이 충분히 크면 평균의 분포는 정규분포에 가까우므로 t-검정을 쓸 수 있습니다.
  - 관심 지표가 고객당 매출의 **평균**이므로 판단은 t-검정으로 합니다.
- 등분산성 검정의 `equal_var`가 False(유의확률 < 0.05)이므로 두 집단의 분산이 같다고 보기 어렵습니다.
  - 가설검정 기초 실습처럼 등분산 가정이 필요 없는 Welch t-검정을 기본으로 사용하고, 등분산성 검정 결과는 참고 정보로 봅니다.
- Levene 검정은 기본적으로 집단 **중위수**와의 거리를 비교합니다.
  - 고객당 매출은 두 집단 모두 중위수가 0원이라, 중위수와의 거리가 매출 그대로가 되어 사실상 평균을 비교하는 검정이 됩니다.
  - 이번 실습에서는 `center='mean'`으로 집단 **평균**과의 거리를 비교합니다.

**질문**: 고객당 평균 매출은 집단 간에 차이가 있는가? Welch t-검정(등분산 가정 필요 없음)과 Student t-검정(등분산 가정 필요)을 비교합니다.

In [ ]:
# A집단의 고객당 매출을 y1에 할당합니다.
y1 = ab_main.loc[ab_main['group'].eq(other='A'), 'revenue_14d']

# B집단의 고객당 매출을 y2에 할당합니다.
y2 = ab_main.loc[ab_main['group'].eq(other='B'), 'revenue_14d']

# 등분산을 가정하지 않는 Welch t-검정을 실행합니다.
# [참고] correction=True는 등분산을 가정하지 않는 독립표본 t-검정을 실행합니다.
pg.ttest(x=y2, y=y1, correction=True)

In [ ]:
# 등분산을 가정한 Student t-검정을 실행합니다.
# [참고] correction=False는 등분산을 가정한 독립표본 t-검정을 실행합니다.
# 기본값 correction='auto'는 두 집단의 표본 크기가 다를 때 Welch t-검정을 실행합니다.
pg.ttest(x=y2, y=y1, correction=False)

**해석**
- 두 검정의 t-통계량과 유의확률이 거의 같습니다.
  - **두 집단의 인원이 같으면** 두 검정의 t-통계량이 수학적으로 같아지고, `dof`(자유도)만 조금 다르기 때문입니다.
- Welch t-검정은 분산이 같을 때도 손해가 거의 없고, 분산이 다를 때는 더 정확하므로 **Welch t-검정을 기본으로 사용합니다.**
  - 유의확률이 0.05보다 작으므로 쿠폰 집단의 고객당 매출이 더 높다고 판단합니다.

**질문**: 순위로 비교하는 Mann-Whitney U 검정에서는 쿠폰 집단의 매출이 더 큰 경향이 있는가?

In [ ]:
# 순위로 비교하는 Mann-Whitney U 검정을 실행합니다.
pg.mwu(x=y2, y=y1)

**해석**
- Mann-Whitney U 검정은 평균이 아니라 **순위**를 비교해 '한 집단의 값이 다른 집단보다 큰 경향이 있는가'를 검정합니다.
  - 평균 차이를 검정하는 t-검정과는 질문이 다르므로, t-검정의 결론을 대신하거나 재확인하는 방법으로 쓰지 않습니다.
- 현재 데이터는 0원 고객이 대부분이라, 사실상 '구매한 고객이 더 많은가'를 비교하는 검정에 가깝습니다.
  - `p_val`(유의확률)이 0.05보다 작으므로 쿠폰 집단의 매출이 순위로 봐도 더 큰 경향이 있습니다.
- 두 집단의 중위수는 모두 0원인데도 Mann-Whitney U 검정은 유의합니다.
  - 순위 검정은 원래 값의 중위수가 아니라, 합친 데이터에서 한 집단의 값이 큰 쪽에 몰려 있는지(평균 순위)를 비교하기 때문입니다.
- 고객당 매출이 관심 지표이므로 판단은 Welch t-검정의 평균 차이와 신뢰구간으로 하고, 이 결과는 보조 정보로 봅니다.

**질문**: 효과 크기는 얼마인가? 고객당 평균 매출 차이와 95% 신뢰구간을 봅니다.

In [ ]:
# 집단별 평균 매출을 출력합니다.
print(f'고객당 평균 매출: A = {y1.mean():,.0f}원, B = {y2.mean():,.0f}원')

In [ ]:
# Welch t-검정 결과에서 평균 차이(B - A)의 95% 신뢰구간을 꺼냅니다.
ci_lower, ci_upper = pg.ttest(x=y2, y=y1, correction=True).loc['T_test', 'CI95']

# 집단별 평균 차이 및 평균 차이의 95% 신뢰구간을 출력합니다.
print(f'차이(B − A) = {y2.mean() - y1.mean():,.0f}원')
print(f'95% 신뢰구간: {ci_lower:,.0f} ~ {ci_upper:,.0f}원')

**해석**
- 독립표본 t-검정의 유의확률은 '차이가 있는가'만 알려 주고, **얼마나** 차이가 나는지는 알려 주지 않습니다.
  - 신뢰구간을 보면 효과의 크기와 불확실성을 함께 알 수 있습니다.
  - 신뢰구간의 하한이 0에 가까울수록 실제 효과가 작을 수도 있다는 것을 의미합니다.
- 이 실험의 본 테스트 인원은 **주 지표인 구매전환율**을 기준으로 정했고, 고객당 매출은 **보조 지표**로 함께 봅니다.
  - 고객당 매출은 0원이 대부분이라 표준편차가 평균보다 훨씬 커서, 같은 검정력을 얻으려면 구매전환율보다 훨씬 많은 인원이 필요합니다.
  - 보조 지표는 유의하지 않게 나와도 '효과가 없다'고 결론 내리지 않습니다.
- 쿠폰 발송 집단의 매출이 늘었다고 해서 회사 이익도 늘었다는 뜻은 아닙니다.
  - 고객당 매출은 쿠폰 할인 전 금액이므로 쿠폰 할인 금액까지 반영한 손익은 5절에서 확인합니다.

### 제1종 오류, 제2종 오류, 검정력

용어의 뜻은 가설검정 실습에서 정리했습니다.

| 용어 | 이 실험에서 |
|---|---|
| 제1종 오류 (α) | 쿠폰 효과가 없는데 쿠폰을 도입하는 실수입니다. 유의수준 0.05로 이 확률을 5%로 묶어 둡니다. |
| 제2종 오류 (β) | 쿠폰 효과가 있는데 놓치는 실수입니다. **파일럿 결과가 바로 이 위험이 큰 상황**이었습니다. |
| 검정력 (1 − β) | 검정력은 실험 전에 정합니다. 본 테스트의 검정력은 0.8 이상입니다. |

- 파일럿에서 '유의하지 않음'이 나왔다고 쿠폰 발송을 포기했다면, 제2종 오류를 범했을 가능성이 큽니다.
- 그래서 실험 **전에** 표본 크기를 계산하고, 실험을 통해 필요한 인원을 모은 뒤 한 번 판단합니다.
- 결과를 보면서 인원을 조금씩 늘리고 유의해질 때 멈추는 방식은 제1종 오류를 키우므로 피해야 합니다.

## 5. 손익 판단

**질문**: 쿠폰 비용까지 고려하면 고객당 이익은 어느 집단이 더 큰가?

In [ ]:
# 마진율을 MARGIN에 할당합니다. 매출의 25%가 마진입니다.
MARGIN = 0.25

# 고객당 마진(매출 × 마진율)을 계산합니다.
ab_main['margin'] = ab_main['revenue_14d'] * MARGIN

# 고객당 이익(마진 - 쿠폰 할인 금액)을 계산합니다.
ab_main['profit'] = ab_main['margin'] - ab_main['discount_amount']

# 집단별 고객당 매출, 마진, 쿠폰비, 이익의 평균을 계산하고 profit_table에 할당합니다.
profit_table = ab_main.groupby(by='group')[['revenue_14d', 'margin', 'discount_amount', 'profit']].mean()

# profit_table의 컬럼명을 '고객당 매출', '고객당 마진', '고객당 쿠폰비', '고객당 이익'으로 변경합니다.
profit_table.columns = ['고객당 매출', '고객당 마진', '고객당 쿠폰비', '고객당 이익']

# profit_table을 소수점 첫째 자리에서 반올림한 결과를 확인합니다.
profit_table.round(decimals=0)

**해석**
- B집단의 고객당 매출과 마진이 A집단보다 높습니다.
  - 회사가 기대하는 고객당 이익은 마진에서 쿠폰 할인 금액을 차감하는 방식으로 계산합니다.
- 고객당 이익은 B집단(1,355원)이 A집단(1,470원)보다 낮습니다.

**질문**: 고객당 이익 차이와, 본 테스트 B집단 4,000명 기준 총 손익은 얼마인가?

In [ ]:
# A집단의 고객당 이익을 p1에 할당합니다.
p1 = ab_main.loc[ab_main['group'].eq(other='A'), 'profit']

# B집단의 고객당 이익을 p2에 할당합니다.
p2 = ab_main.loc[ab_main['group'].eq(other='B'), 'profit']

# 고객당 이익 차이(B - A)를 계산합니다.
profit_diff = p2.mean() - p1.mean()

# 고객당 이익 차이의 Welch t-검정을 실행하고 profit_ttest에 할당합니다.
profit_ttest = pg.ttest(x=p2, y=p1, correction=True)

# Welch t-검정 결과를 확인합니다.
profit_ttest

In [ ]:
# Welch t-검정 결과에서 고객당 이익 차이(B - A)의 95% 신뢰구간을 꺼냅니다.
ci_lower, ci_upper = profit_ttest.loc['T_test', 'CI95']

# 고객당 이익 차이 및 95% 신뢰구간을 출력합니다.
print(f'고객당 이익 차이(B − A) = {profit_diff:,.0f}원')
print(f'95% 신뢰구간: {ci_lower:,.0f} ~ {ci_upper:,.0f}원')

In [ ]:
# B집단 인원 기준 총 손익을 계산합니다.
print(f'B집단 {p2.shape[0]:,}명 기준 쿠폰 시행에 따른 총 손익 = {profit_diff * p2.shape[0]:,.0f}원')

**해석**
- 고객당 이익 차이의 점추정값이 음수입니다.
  - 쿠폰 발송이 회사 이익을 줄이는 방향으로 추정됩니다.
- 고객당 이익 차이의 신뢰구간이 0을 포함하므로 **이익이 늘었다는 근거는 전혀 없습니다.**
  - 물론 '손해가 확실하다'고까지 말할 수도 없습니다.
  - 쿠폰 도입을 정당화하려면 고객당 이익이 늘었다는 증거가 필요합니다.
- 이번 실습에서는 쿠폰 발송이 **구매전환율과 매출을 통계적으로 유의하게 높였습니다.**
  - 마진율을 반영한 고객당 이익에서는 **늘었다는 근거가 없고, 오히려 줄어드는 방향**입니다.
- '통계적으로 유의하다'는 '차이가 우연이 아니다'라는 뜻일 뿐이며 '그 차이가 비용을 넘을 만큼 크다'는 뜻은 아닙니다.
  - 의사결정 과정에서 유의확률과 함께 **효과 크기를 돈으로 환산한 값**을 확인할 필요가 있습니다.
  - 현업에서는 쿠폰 금액이나 사용 조건(최소 구매 금액)을 바꿔서 여러 번 실험하는 방식을 사용합니다.
- 물론 고객당 이익만이 의사결정 기준은 아닙니다.
  - 신규 고객 확보, 시장 점유율 확대, 재고 소진처럼 **이익을 일부 포기하더라도 매출 규모를 키워야 하는 상황**도 있습니다.
  - 이때는 이익 감소분을 '구매 고객 확보에 든 비용'으로 보고, 늘어난 구매 고객 1명당 비용이 회사가 감수할 수 있는 수준인지 판단합니다.
  - 또한 이번 실험은 14일 동안의 이익만 보았으므로 쿠폰으로 구매한 고객이 이후에 재구매한다면 장기적으로는 이익이 될 수도 있습니다.

## 정리

- 카이제곱 검정으로 두 집단의 고객 구성이 비슷함을 확인했습니다.
  - 배정에 문제가 있다는 신호는 없었지만, 이 비교만으로 무작위 배정을 증명할 수는 없습니다.
- 파일럿은 인원이 적어 검정력이 낮았고, 구매전환율 차이를 확인할 수 없었습니다.
  - 표본 크기를 계산해 본 테스트 인원을 정했습니다.
- 본 테스트에서는 구매전환율과 고객당 매출 모두 쿠폰 집단이 유의하게 높았습니다.
  - 모비율 차이 검정과 Welch t-검정으로 판단했고, 순위로 비교한 Mann-Whitney U 검정은 보조 정보로 봤습니다.
- 마진율과 쿠폰 비용을 반영하면 고객당 이익은 쿠폰 발송 집단이 더 낮았습니다.
  - 통계적 유의성과 사업적 이익은 별개입니다.

## 현업 전달 포인트

통계 용어 없이 설명하면 다음과 같습니다.

- 쿠폰을 보내면 구매하는 고객과 매출이 늘어난다는 것을 실험으로 확인했습니다.
  - 고객당 이익은 약 115원 줄어드는 방향으로 추정되고 늘었다는 근거는 없습니다.
  - 지금 조건을 전체 고객으로 확대하는 것은 보류하는 것이 좋습니다.
- 처음 파일럿 테스트에서는 인원이 부족해 쿠폰 발송에 따른 구매 증가 효과를 판단할 수 없었습니다.
  - 필요한 인원을 계산해 본 테스트를 실행한 결과입니다.
- 쿠폰 금액이나 최소 구매 금액 조건을 바꾼 안으로 다시 테스트해 볼 것을 제안합니다.

## End of Document